# COMP1818 Artificial Intelligence Applications - Practical Implementation
## Retail Loan Approval Prediction: Benchmarking Simple Baselines & Explainable Ensembles
**Course:** COMP1818 Artificial Intelligence Applications, University of Greenwich  
**Author:** Kirtan Karkar  
**Anchor Paper:** Gawande et al. (2026) *An Explainable Ensemble Learning Framework for Transparent and Accurate Loan Approval Prediction*, Discover Artificial Intelligence.  
**Dataset:** Kaggle Loan Prediction Problem Dataset (=614$)  
---
### Notebook Structure:
1. **Exploratory Data Analysis (EDA):** Dataset inspection, class balance, and missing value profiling.
2. **Leak-Free Preprocessing:** Pipelines with  (median imputation, standardization, one-hot encoding).
3. **Baseline Models (LO3):** Majority class, random chance, and symbolic rule engines.
4. **Supervised Machine Learning:** Logistic Regression, Decision Tree, Random Forest, SVM, and XGBoost.
5. **Proposed Extension:** Random Forest + XGBoost soft-voting ensemble.
6. **Nested Stratified Cross-Validation:** Out-of-sample generalization variance ( 	imes 3$).
7. **Model Interpretability & SHAP:** Global feature attribution.
8. **Asymmetric Lending Risk Analysis (LO2):** False Approval vs False Rejection cost threshold optimization.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append(os.path.abspath(".."))
from src.data_loader import get_data_splits, build_preprocessor
from src.models import build_model_pipelines, get_hyperparameter_grids
from src.evaluate import run_nested_cross_validation, evaluate_on_test_set, merge_and_save_results
from src.explainability import compute_feature_importance, analyze_asymmetric_lending_costs

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

## 1. Data Ingestion and Profiling

In [2]:
X_train, X_test, y_train, y_test, num_cols, cat_cols = get_data_splits(
    filepath="../data/train.csv", test_size=0.20, random_state=42
)
print(f"Training instances: {len(X_train)} (Approved: {(y_train==1).sum()}, Rejected: {(y_train==0).sum()})")
print(f"Testing instances:  {len(X_test)} (Approved: {(y_test==1).sum()}, Rejected: {(y_test==0).sum()})")
print("Numerical Features:", num_cols)
print("Categorical Features:", cat_cols)
X_train.head()

## 2. Initialize Model Pipelines & Baselines (LO3)

In [3]:
preprocessor = build_preprocessor(num_cols, cat_cols)
models = build_model_pipelines(preprocessor, random_state=42)
grids = get_hyperparameter_grids()
print(f"Constructed {len(models)} model pipelines across 4 distinct paradigms.")

## 3. Nested Stratified 5-Fold Cross-Validation

In [4]:
cv_summary, fold_scores, fitted_models = run_nested_cross_validation(
    models=models, param_grids=grids, X_train=X_train, y_train=y_train, outer_folds=5, inner_folds=3
)
cv_summary

## 4. Evaluation on Independent Test Set (=123$)

In [5]:
test_summary, preds_dict, probs_dict = evaluate_on_test_set(
    fitted_models=fitted_models, X_test=X_test, y_test=y_test
)
results_df = merge_and_save_results(cv_summary, test_summary, fold_scores, output_dir="../outputs")
results_df[["Model", "Test Accuracy", "Test Macro F1", "Test ROC AUC", "CV Macro F1 Mean", "CV Macro F1 Std"]]

## 5. Explainable AI: Feature Attribution (SHAP) & Asymmetric Lending Costs (LO2)

In [6]:
importance_df, shap_vals, clean_names, _ = compute_feature_importance(
    model=fitted_models["RF + XGBoost Soft Voting"],
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    output_dir="../outputs"
)
print("Top 10 Feature Importances:")
print(importance_df.head(10))

ens_probs = probs_dict["RF + XGBoost Soft Voting"]
thresholds, costs, opt_threshold = analyze_asymmetric_lending_costs(
    y_true=y_test.to_numpy(), y_prob=ens_probs, cost_fp=5.0, cost_fn=1.0
)
print(f"Risk-Optimal Classification Threshold: τ* = {opt_threshold:.2f}")